# Mestrado em Informática
## Aprendizagem Máquina
Ano letivo 2025/26 - Trabalho Prático 2

**Miguel de Campos Rodrigues - 27089**

O objetivo deste trabalho é utilizar algoritmos de classificação para treinar modelos preditivos da situação académica de estudantes de cursos de Licenciatura do Instituto Politécnico de Portalegre.  O conjunto de dados anonimizados encontra-se no ficheiro **dataset_PIAES_reduced.xlsx**.

As características presentes no conjunto de dados dizem respeito a uma parte da informação existente sobre os estudantes no ato da matrícula, nomeadamente:
`Sexo` (M/F)
`Estado civil` (desc)
`Ordem de ingresso` (opção de preferência de ingresso no curso – 1 representa 1ª opção)
`Curso` (desc)
`Deslocado` (S/N)
`Nacionalidade` (desc)
`Habilitação literária mãe` (desc)
`Situação profissional mãe` (desc)
`Situação profissional pai` (desc)
`Nota ingresso`
`Bolseiro` (1 para detentor de bolsa, 0 para o caso contrário)
`Idade no ingresso`

O campo Categoria tem como valores possíveis Diplomado / Não Diplomado e indica se o estudante obteve ou não o seu diploma ao fim de N anos de frequência do curso de Licenciatura. N representa a duração de um ciclo de estudos, e tem valor 4 para a Licenciatura em Enfermagem, valor 3 para todos os outros cursos de Licenciatura do IPP.

Na categoria “Não Diplomado” estão incluídos os estudantes que, estando inscritos, ainda não terminaram os seus cursos ao fim de N anos, e também os estudantes que abandonaram o curso ao longo do seu percurso académico.

Pretende-se, portanto, treinar modelos de aprendizagem máquina com o objetivo de prever, o mais cedo possível no percurso académico dos estudantes, a sua situação ao fim de N anos.

Aspetos a ter em conta no desenvolvimento do trabalho:
<ol type="A">
  <li>Pré-processamento dos dados, justificando as opções adotadas.</li>
  <li>Treino/ teste de modelos e otimização de parâmetros para os algoritmos de classificação estudados (incluindo regressão logística, k vizinhos mais próximos, máquinas de vetor de suporte, árvores de decisão, florestas aleatórias e redes neuronais).</li>
  <li>Análise da importância das características.</li>
  <li>Comparação dos modelos treinados e eleição justificada do(s) modelo(s) mais adequado(s).</li>
</ol>

O trabalho deve ser entregue pelo PAE no formato Jupyter Lab (*.ipynb), com o código, as justificações e a discussão final.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

%matplotlib inline
sns.set_style("whitegrid")

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, accuracy_score
from sklearn.impute import SimpleImputer


## Pré-processamento dos dados
Primeiro iremos ler carregar o dataset.

In [2]:
df=pd.read_excel("dataset_PIAES_reduced.xlsx")
print("Shape:", df.shape)
df.columns

Shape: (5742, 13)


Index(['Sexo (M/F)', 'Estado civil (desc)', 'Ordem de ingresso',
       'Curso (desc)', 'Deslocado (S/N)', 'Nacionalidade (desc)',
       'Habilitação literária mãe (desc)', 'Situação profissional mãe (desc)',
       'Situação profissional pai (desc)', 'Nota ingresso', 'Bolseiro',
       'Idade no ingresso', 'Categoria'],
      dtype='object')

O dataset contém 5742 registos e 13 colunas. Com base na análise feita na unidade curricular de Programação de Ciência de Dados será feito o mesmo tratamento de dados:

1. Remoção de outliers na nota de ingresso: $Notas < 95 \land Notas > 200$
2. Imputação de dados nulos usando o SimpleImputer:
   1. Para colunas numéricas usa-se a mediana,
   2. Na categórica usa-se a categoria mais comum.
3. Normalização das habilitações literárias
4. Criação de colunas extras:
   1. Escola
   2. Curso
   2. Nr. anos de curso
   3. Família de Risco
   4. Indicadores geodemográficos
5. Numeração de classificadores através de `OneHotEncoder` para classificadores  binários e `LabelEncoder` para outros classificadores com um grande número de entradas
7. Normaliação de todos os classificadores numéricos:
   1. Nota normalizada utilizando a transformação `StandardScaler`, para comparação relativa entre o desempenho das notas:
   2. Idade normalizada utilizando a transformação `RobustScaler`, pois quanto mais velhos mais se tornam outliers.

In [3]:
def remove_outliers():
  global df
  df = df[df["Nota ingresso"] >= 95]
  df = df[df["Nota ingresso"] < 200]

def impute_null_values():
  global df
  medianImputer = SimpleImputer(strategy='median')
  commonImputer = SimpleImputer(strategy='most_frequent')

  df_num = df.select_dtypes(include='number')
  df_num_imp = pd.DataFrame(
      medianImputer.fit_transform(df_num),
      columns=df_num.columns,
      index=df.index)

  df_hab_mae = df[['Habilitação literária mãe (desc)']]
  df_hab_mae_imp = pd.DataFrame(
      commonImputer.fit_transform(df_hab_mae),
      columns=df_hab_mae.columns,
      index=df.index)

  df[df_num_imp.columns] = df_num_imp
  df[df_hab_mae_imp.columns] = df_hab_mae_imp

def normalize_mother_degree():
  global df
  # Habilitação literária mãe (desc)
  col_hab = "Habilitação literária mãe (desc)"
  habilitacoes = {
      "Sem escolaridade obrigatória": [
          "Sabe ler sem possuir o 4º ano de escolaridade",
          "Não sabe ler nem escrever"
      ],
      "Ensino Básico 2º Ciclo (6º/7º/8º Ano) ou Equiv.": [
          "7º Ano de Escolaridade",
          "9º Ano de Escolaridade - Não Concluído",
          "7º Ano (Antigo)", "8º Ano de Escolaridade"
      ],
      "Ensino Básico 3º Ciclo (9º/10º/11º Ano) ou Equiv.": [
          "Curso Geral de Administração e Comércio",
          "Outro - 11º Ano de Escolaridade",
          "11º Ano de Escolaridade - Não Concluído",
          "10º Ano de Escolaridade",
          "12º Ano de Escolaridade - Não Concluído",
          "Curso Geral de Comércio",
          "Curso Complementar Liceal - Não Concluído"
      ],
      "Ensino Secundário (12º Ano) ou Equiv.": [
          'Curso Técnico-Profissional',
          'Curso Complementar Liceal',
          '2ª Ciclo do Curso Geral Liceal',
          'Complementar de Contabilidade e Administração',
          '2º Ano Curso Complementar Liceal'
      ],

      "Cursos superiores ou de especialização (QEQ Nível 5)": [
          "Curso de Especialização Tecnológica",
          "Curso Técnico Superior Profissional",
          "Curso de Estudos Superiores Especializados",
          "Frequência do Ensino Superior",
      ],

      "Ensino Superior - Licenciatura":[
          "Ensino Superior - Licenciatura (1º ciclo)"
      ],

      "Ensino Superior - Mestrado": [
          "Ensino Superior - Mestrado (2º ciclo)"
      ],

      "Ensino Superior - Doutoramento": [
          "Ensino Superior - Doutoramento (3º ciclo)"
      ]
  }

  for key, items in habilitacoes.items():
      df.loc[df[col_hab].isin(items), col_hab] = key

  # Classificação das habilitações Literárias
  ordered_hab = [
      'Desconhecido',
      'Sem escolaridade obrigatória',
      'Ensino Básico 1º Ciclo (4º/5º Ano) ou Equiv.',
      'Ensino Básico 2º Ciclo (6º/7º/8º Ano) ou Equiv.',
      'Ensino Básico 3º Ciclo (9º/10º/11º Ano) ou Equiv.',
      'Ensino Secundário (12º Ano) ou Equiv.',
      'Cursos superiores ou de especialização (QEQ Nível 5)',
      'Ensino Superior - Licenciatura',
      'Ensino Superior - Bacharelato',
      'Ensino Superior - Mestrado',
      'Ensino Superior - Doutoramento',
  ]

  col_hab_desc = "Habilitação literária mãe (desc)"
  col_hab_class = "Habilitação literária mãe (classe)"

  for i, habilitacao in enumerate(ordered_hab):
      mask = df[col_hab_desc] == habilitacao
      df.loc[mask, col_hab_class] = i

def normalize_school():
  global df
  # Escolas
  escolas = {
      'ESS - Escola Superior de Saúde': [
          'Enfermagem',
          'Higiene Oral',
      ],
      'ESTGD - Escola Superior de Tecnologias, Gestão e Design': [
          'Gestão',
          'Gestão (pós-laboral)',
          'Administração de Publicidade e Marketing',
          'Design de Animação e Multimédia',
          'Engenharia Informática',
          'Design de Comunicação',
          'Tecnologias de Produção de Biocombustíveis',
      ],
      'ESECS - Escola Superior de Educação e Ciencias Sociais': [
          'Turismo',
          'Jornalismo e Comunicação',
          'Serviço Social',
          'Serviço Social (pós-laboral)',
          'Educação Básica',
      ],
      'ESBE - Escola Superior de Biociências de Elvas': [
          'Agronomia',
          'Equinicultura',
          'Enfermagem Veterinária',
      ],
  }

  col_escola_desc = "Escola (desc)"
  col_escola_class = "Escola (classe)"
  col_curso_desc = "Curso (desc)"

  for i, (escola, cursos) in enumerate(escolas.items()):
      curso_mask = df[col_curso_desc].isin(cursos)
      # df.loc[curso_mask, col_escola_desc] = escola
      df.loc[curso_mask, col_escola_class] = i

def normalize_courses():
  global df
  # Curso (desc), Estado civil (desc), Situação profissional mãe (desc), Situação profissional pai (desc)
  query_cols = [
      'Curso (desc)',
      'Estado civil (desc)',
      'Situação profissional mãe (desc)',
      'Situação profissional pai (desc)']
  novas_cols = [
      'Curso (classe)',
      'Estado civil (classe)',
      'Situação profissional mãe (classe)',
      'Situação profissional pai (classe)']

  for col, new_col in zip(query_cols, novas_cols):
      le = LabelEncoder()
      df[new_col] = le.fit_transform(df[col])

def binary_indicators():
  global df
  # indicadores binários OneHotEncoder
  ohe=OneHotEncoder(sparse_output=False, drop="first")
  query_cols = ["Sexo (M/F)", "Deslocado (S/N)", "Categoria"]
  new_vals = ohe.fit_transform(df[query_cols])
  new_cols = ohe.get_feature_names_out()

  df_ohe = pd.DataFrame(new_vals, columns = new_cols, index = df.index)
  df = pd.concat([df, df_ohe], axis=1)

  # indicadores binários (manuais)
  rendimento_estavel = [
      "Trabalhador(a) por Conta Própria como Isolado",
      "Trabalhador(a) por Conta Própria como Empregador",
      "Trabalhador(a) por Conta de Outrém",
      "Serviço Militar"
  ]

  df['Família de risco'] = (
      ~df['Situação profissional mãe (desc)'].isin(rendimento_estavel) &
      ~df['Situação profissional pai (desc)'].isin(rendimento_estavel)
  ).astype(int)

  origem_acordo = ["Portuguesa", "Brasileira", "Guineense", "Santomense", "Caboverdeana", "Angolana", "Moçambicana"]
  origem_ue = ["Portuguesa", "Espanhola", "Romena", "Italiana", "Alemã", "Holandesa", "Bulgara", "Lituana"]
  origem_america = ["Brasileira", "Mexicana", "Cubana", "Uruguai"]
  origem_euroasiatica = ["Portuguesa", "Espanhola", "Ucraniana", "Romena", "Italiana",
                        "Moldova (República de)", "Alemã", "Russa", "Holandesa", "Turca", "Bulgara",
                        "Inglesa", "Lituana"]
  origem_africana = ["Guineense", "Santomense", "Caboverdeana", "Angolana", "Moçambicana"]

  df['Origem de país de acordo ortográfico'] = df['Nacionalidade (desc)'].isin(origem_acordo).astype(int)
  df['Origem de país da UE'] = df['Nacionalidade (desc)'].isin(origem_ue).astype(int)
  df['Origem de país do continente americano'] = df['Nacionalidade (desc)'].isin(origem_america).astype(int)
  df['Origem de país estrangeiro'] = (df['Nacionalidade (desc)'] != "Portuguesa").astype(int)
  df['Origem de país do continente euroásiatico'] = df['Nacionalidade (desc)'].isin(origem_euroasiatica).astype(int)
  df['Origem de país do continente africano'] = df['Nacionalidade (desc)'].isin(origem_africana).astype(int)

def numeric_indicators():
  global df
  # indicadores numéricos
  df['Nr. anos de curso'] = (df['Curso (classe)'] == 5).astype(int) + 3
  df['Nr. anos de curso'].value_counts()
  df['Nota ingresso'] = StandardScaler().fit_transform(df[['Nota ingresso']])
  df['Idade no ingresso'] = RobustScaler().fit_transform(df[['Idade no ingresso']])

def drop_unused():
  global df
  df = df.drop([
       'Sexo (M/F)', 'Estado civil (desc)', 'Curso (desc)', 'Deslocado (S/N)',
       'Nacionalidade (desc)', 'Habilitação literária mãe (desc)',
       'Situação profissional mãe (desc)', 'Situação profissional pai (desc)',
       'Categoria'], axis=1)

In [4]:
remove_outliers()
impute_null_values()
normalize_mother_degree()
normalize_school()
normalize_courses()
numeric_indicators()
binary_indicators()
drop_unused()

df.columns

Index(['Ordem de ingresso', 'Nota ingresso', 'Bolseiro', 'Idade no ingresso',
       'Habilitação literária mãe (classe)', 'Escola (classe)',
       'Curso (classe)', 'Estado civil (classe)',
       'Situação profissional mãe (classe)',
       'Situação profissional pai (classe)', 'Nr. anos de curso',
       'Sexo (M/F)_M', 'Deslocado (S/N)_S', 'Categoria_Não Diplomado',
       'Família de risco', 'Origem de país de acordo ortográfico',
       'Origem de país da UE', 'Origem de país do continente americano',
       'Origem de país estrangeiro',
       'Origem de país do continente euroásiatico',
       'Origem de país do continente africano'],
      dtype='object')

In [5]:
df

,Ordem de ingresso,Nota ingresso,Bolseiro,Idade no ingresso,Habilitação literária mãe (classe),Escola (classe),Curso (classe),Estado civil (classe),Situação profissional mãe (classe),Situação profissional pai (classe),...,Sexo (M/F)_M,Deslocado (S/N)_S,Categoria_Não Diplomado,Família de risco,Origem de país de acordo ortográfico,Origem de país da UE,Origem de país do continente americano,Origem de país estrangeiro,Origem de país do continente euroásiatico,Origem de país do continente africano
0,5.0,0.035224,0.0,0.000000,4.0,1.0,2,4,9,4,...,1.0,1.0,1.0,0,1,1,0,0,1,0
1,1.0,1.073900,0.0,-0.166667,7.0,2.0,16,4,9,10,...,1.0,1.0,0.0,0,1,1,0,0,1,0
2,5.0,-0.135611,0.0,-0.166667,2.0,1.0,3,4,9,2,...,1.0,1.0,1.0,0,1,1,0,0,1,0
3,2.0,-0.490947,0.0,0.000000,2.0,2.0,12,4,9,10,...,0.0,1.0,0.0,0,1,1,0,0,1,0
4,1.0,1.005566,0.0,4.166667,3.0,2.0,14,0,5,5,...,0.0,0.0,0.0,1,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5737,2.0,-0.531947,0.0,-0.333333,5.0,2.0,12,4,7,10,...,0.0,1.0,1.0,0,0,0,0,1,1,0
5738,1.0,-0.736949,1.0,0.000000,5.0,0.0,5,4,9,10,...,0.0,0.0,0.0,0,1,1,0,0,1,0
5739,1.0,1.552237,1.0,1.666667,2.0,0.0,5,4,5,2,...,0.0,1.0,1.0,1,1,1,0,0,1,0
5740,1.0,1.846073,1.0,0.000000,2.0,1.0,9,4,9,10,...,0.0,1.0,0.0,0,1,1,0,0,1,0


## Treino / teste de modelos e otimização de parâmetros